# Attention

The `MultiheadAttention` was published in 2017 in the paper [Attention is All You Need](https://arxiv.org/abs/1706.03762) by Google Brain. It is a key component of the Transformer and has gained widespread attention (no pun intended) in diverse architectures ranging from computer vision to natural language processing and bioinformatics. 

## ScaledDot-Product Attention

The attention function is described as answering a query $q_t$ with a key-value pair $k_t$, $v_t$ and subsequently mapping them to an output $o_t$ for the $t$-th token. The output $o_t$ is a weighted sum of $v_t$ where the weight is a *compatibility* function between the query and respective key. 

<img src="/Users/otto/attention/attention/images/architecture.png" alt="drawing" width="600"/>


## Computation


Take the sentence "The dog jumped over the box". These are tokenized to become [$x_{the}$, $x_{dog}$, ... ,   $x_{box}$], where $x_t \in \mathbb{R}^d$

1. **Make query** $q_t$, by linear transformation with $W^{Q} \in \mathbb{R}^{d_{model} \times d_k}$ which is learned during training, and $d_k$ is the dimension of query/key/value vectors. The resulting query is 
$$q_t = W^Q x_t$$

2. **Make keys and values** with $W^K \in \mathbb{R}^{d_{model} \times d_k}$ and $W^V \in \mathbb{R}^{d_{model} \times d_k}$ respectively. 

$$k_t = W^K x_t \text{ and } v_t = W^V x_t$$

3. **Compute the amount of attention token $t$ pays to each other token**. Denote the *amount* of attention paid by $score_i$ for each other token $t \in \{1, 2, ... T\}$ where $i$ is the number of tokens. The score is computed as

$$score_i = q_t \cdot k_i \text{ for each } i,$$ 

and represents the compatibility between the question $q_t$ and the summary $k_i$ of information between token $t$ and $i$.

4. **Calculate the weighted sum** $o_t$ for each token $t$ as 

$$o_t = \sum_i^{T} \text{softmax}(q_t \cdot k_i) \cdot v_i,$$

where $t \in \{1, 2, ... , T\}$ and $T$ is the total number of tokens in a sequence. 

## Implementation

The example sentence "The dog jumped over the box" is chosen. $d_{model}$ is set to 16 and $d_k$ is set to 8 (in code as $\lfloor d_{model} \rfloor$). The words are tokenized


In [18]:
sentence = "The dog jumped over the box"
# Create a vocabulary and map tokens to indices
tokens = sentence.lower().split()
T = len(tokens)

d_model = 16
d_k = d_model // 2 

In [ ]:
import torch

def tokenizer(sentence):
    tokens = sentence.lower().split()
    vocab = {word: idx for idx, word in enumerate(set(tokens))}
    token_indices = [vocab[token] for token in tokens]
    tokens_tensor = torch.tensor(token_indices, dtype=torch.long)
    return tokens_tensor, vocab

In [ ]:
print("d_model:", d_model)
print("d_k:", d_k)
print("sentence:", sentence)
print("tokens:", tokens)
print("T:", T)

d_model: 16
d_k: 8
sentence: The dog jumped over the box
tokens: ['the', 'dog', 'jumped', 'over', 'the', 'box']
T: 6


## Embedding Layer

The purpose of this is to convert the input tokens into a dense vector representation. The embedding layer is a lookup table that maps each token to a vector of fixed size. It is typically initialized with random weights and trained along with the rest of the model.

In this implementation, `nn.Embedding` class from PyTorch is used to initialize the embedding layer with a dictionary of 10 and 3 dimensions.

In [ ]:
# The embedding layer is initialized with 10 words and 3 dimensions
# The embedding layer is initialized with random weights
embedding = torch.nn.Embedding(10, 3)
embeddings = embedding()

## Interpretations and Intuition

**Interpretations** of each value
- $q_t$ is the query vector for the $t$-th token. It represents the information that token $t$ is *looking* for.
- $k_t$ is the key vector for the $t$-th token and represents what this token $contains$ or can what it can $answer$ to.
- $v_t$ is the value vector for the $t$-th token and it has the relevant information that the key unlocks if the query deems this token relevant

**Example**

E.g., in the sentence "the dog jumped over the box" the query $q_{jumped}$ pulls information from the key-value pairs of "dog" and "box" the most because "jumped" is trying to understand *who* jumped and *to where*. $q_{jumped}$ is the question "jumped" is asking, $k_{jumped}$ is the reference point, and $v_{jumped}$ is what "jumped" contributes if chosen. 


**Intuition/rule of thumb**
- Query: What am I looking for?
- Key: What do I have to answer queries? (Summary of information in value).
- Value: What do I share if my knowledge is to be unlocked?

The last step (4.) should enforce the intuition. The softmax function

$$ \text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{T} e^{x_j}} $$

normalizes a vector and transforms it into a probability distribution, e.g., 

$$ \text{softmax}([1, 2, 3]) = [0.09003057, 0.66524096, 0.24472847] $$

So, when softmax is computed over $q_t \cdot k_i$, it is encompassing the probability of each token $i$ being relevant to the query $q_t$. The weighted sum of the values $v_i$ is then a summary of the information that is relevant to the query $q_t$. **The weighted sum is thus the value of each token $i$ multiplied by the probability of it being relevant to the query $q_t$.**